# VoxForge — F5-TTS Việt 6000h GPU Server (Miễn phí 100% trên Google Colab / Kaggle)

Notebook này chạy mô hình **F5-TTS Việt Nam 6.000 giờ (`hynt/F5-TTS-Vietnamese-ViVoice/model_last_6000h.pt`)** trực tiếp trên GPU miễn phí (NVIDIA T4 16GB) và tạo đường hầm HTTPS miễn phí qua **Cloudflare Tunnel (`trycloudflare.com`)** để kết nối với website **VoxForge Studio** của bạn.

### Hướng dẫn chạy trong 2 phút:
1. Trên thanh menu Google Colab: chọn **Runtime (Thời gian chạy) → Change runtime type → T4 GPU → Save**.
2. Bấm **Runtime → Run all** (hoặc bấm nút ▶ ở từng ô bên dưới).
3. Chờ ~2–3 phút tải mô hình (chỉ tải 1 lần trong phiên), sau đó **copy đường link `https://....trycloudflare.com`** hiện ra ở ô số 2.
4. Dán link đó vào mục **Mô hình AI → GPU Server riêng (Google Colab)** trên website VoxForge của bạn và bấm **Lưu & Kiểm tra GPU**.
5. Quay lại **Tạo giọng nói** và chuyển văn bản dài **không giới hạn ký tự** (10.000 – 100.000+ ký tự) không bao giờ gặp lỗi ZeroGPU!

In [ ]:
#@title 1. Cài đặt F5-TTS Việt 6000h & Cloudflare Tunnel (chạy 1 lần, ~2 phút)
import os, subprocess, torch

assert torch.cuda.is_available(), (
    '❌ Chưa bật GPU! Hãy vào menu Runtime -> Change runtime type -> chọn T4 GPU rồi chạy lại.'
)
print('✅ GPU phát hiện:', torch.cuda.get_device_name(0))

# Cài đặt thư viện cần thiết
!pip install -q f5-tts fastapi uvicorn python-multipart soundfile cached-path huggingface-hub
!pip install -q vinorm || true

# Tải cloudflared (Cloudflare Quick Tunnel miễn phí, không cần tài khoản)
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

print('✅ Cài đặt hoàn tất! Đang chuẩn bị tải trọng số F5-TTS Việt 6000h ở ô tiếp theo...')

In [ ]:
#@title 2. Khởi chạy F5-TTS Việt 6000h GPU Server & Lấy link dán vào VoxForge
import base64
import hashlib
import io
import os
import re
import subprocess
import tempfile
import threading
import time
from typing import Optional

import numpy as np
import soundfile as sf
import torch
import uvicorn
from cached_path import cached_path
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from f5_tts.model import DiT
from f5_tts.infer.utils_infer import (
    infer_process,
    load_model,
    load_vocoder,
    preprocess_ref_audio_text,
)

try:
    from vinorm import TTSnorm
except Exception:
    TTSnorm = None

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'⏳ Đang nạp mô hình F5-TTS Việt 6000h lên {GPU_NAME}...')

vocoder = load_vocoder()
model = load_model(
    DiT,
    dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4),
    ckpt_path=str(cached_path('hf://hynt/F5-TTS-Vietnamese-ViVoice/model_last_6000h.pt')),
    vocab_file=str(cached_path('hf://hynt/F5-TTS-Vietnamese-ViVoice/config_6000h.json')),
)
print('✅ Đã nạp xong mô hình F5-TTS Việt 6000h lên GPU!')

# Cache giọng mẫu theo SHA-256 để chỉ chạy ASR (nhận diện lời mẫu) ĐÚNG 1 LẦN cho cả cuốn truyện!
REF_DIR = tempfile.mkdtemp(prefix='voxforge_refs_')
REF_CACHE = {}  # sha256 -> {'path': str, 'ref_text': str}
INFER_LOCK = threading.Lock()


def post_process_vn(text: str) -> str:
    if TTSnorm is not None:
        try:
            text = TTSnorm(text)
        except Exception:
            pass
    text = ' ' + text + ' '
    for old, new in [(' . . ', ' . '), (' .. ', ' . '), (' , , ', ' , '), (' ,, ', ' , '), ('"', '')]:
        text = text.replace(old, new)
    return ' '.join(text.split()).lower()


class SynthesizeRequest(BaseModel):
    text: str
    ref_audio_base64: Optional[str] = None
    ref_hash: Optional[str] = None
    ref_text: Optional[str] = ''
    speed: float = 1.0
    nfe_step: int = 32
    cfg_strength: float = 2.0
    cross_fade_duration: float = 0.15


app = FastAPI(title='VoxForge F5-TTS Vietnamese 6000h GPU Server')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)


@app.get('/health')
def health():
    return {
        'status': 'ok',
        'model': 'F5-TTS-Vietnamese-6000h',
        'gpu': GPU_NAME,
        'cached_voices': len(REF_CACHE),
    }


@app.post('/synthesize')
def synthesize(req: SynthesizeRequest):
    if not req.text or not req.text.strip():
        raise HTTPException(status_code=400, detail='Văn bản trống.')

    with INFER_LOCK:
        ref_key = req.ref_hash
        cached = REF_CACHE.get(ref_key) if ref_key else None

        if cached is None:
            if not req.ref_audio_base64:
                raise HTTPException(status_code=404, detail='REF_CACHE_MISS')
            raw_bytes = base64.b64decode(req.ref_audio_base64)
            ref_key = hashlib.sha256(raw_bytes).hexdigest()[:24]
            cached = REF_CACHE.get(ref_key)
            if cached is None:
                raw_path = os.path.join(REF_DIR, f'{ref_key}.wav')
                with open(raw_path, 'wb') as f:
                    f.write(raw_bytes)
                ref_audio_path, detected_ref_text = preprocess_ref_audio_text(
                    raw_path, (req.ref_text or '').strip()
                )
                cached = {
                    'path': ref_audio_path,
                    'ref_text': (detected_ref_text or '').strip().lower(),
                }
                REF_CACHE[ref_key] = cached
                print(f'🎙️ Đã cache giọng mẫu [{ref_key}]: "{cached["ref_text"][:80]}"')

        clean_gen_text = post_process_vn(req.text)
        speed = max(0.5, min(1.8, float(req.speed or 1.0)))
        nfe_step = max(16, min(64, int(req.nfe_step or 32)))

        t0 = time.time()
        final_wave, sr, _ = infer_process(
            cached['path'],
            cached['ref_text'],
            clean_gen_text,
            model,
            vocoder,
            speed=speed,
            nfe_step=nfe_step,
            cfg_strength=float(req.cfg_strength or 2.0),
            cross_fade_duration=float(req.cross_fade_duration or 0.15),
        )
        elapsed = round(time.time() - t0, 2)

        wav_io = io.BytesIO()
        sf.write(wav_io, np.asarray(final_wave, dtype=np.float32), sr, format='WAV', subtype='PCM_16')
        wav_bytes = wav_io.getvalue()
        dur_sec = round(len(final_wave) / float(sr), 2)
        print(f'✅ Tạo xong {len(req.text)} ký tự -> {dur_sec}s audio trong {elapsed}s trên {GPU_NAME}')

        return {
            'audio_base64': base64.b64encode(wav_bytes).decode('ascii'),
            'mime_type': 'audio/wav',
            'duration_sec': dur_sec,
            'ref_hash': ref_key,
            'gpu': GPU_NAME,
        }


# Khởi chạy FastAPI ở luồng nền
def run_server():
    uvicorn.run(app, host='127.0.0.1', port=7860, log_level='warning')


threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

# Mở Cloudflare Quick Tunnel miễn phí
tunnel_proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match and not public_url:
        public_url = match.group(0)
        print('\n' + '=' * 66)
        print(f'🚀 F5-TTS VIỆT 6000H GPU SERVER ({GPU_NAME}) ĐÃ SẴN SÀNG!')
        print('📋 Copy đường link dưới đây dán vào mục "Mô hình AI" trên VoxForge:')
        print(f'\n   {public_url}\n')
        print('⚠️ Giữ tab Google Colab này mở trong lúc tạo giọng nói.')
        print('=' * 66 + '\n')
        break

# Giữ ô chạy liên tục và xả log tunnel để không bao giờ bị nghẽn bộ đệm
try:
    for _ in tunnel_proc.stdout:
        pass
    tunnel_proc.wait()
except KeyboardInterrupt:
    tunnel_proc.terminate()
    print('Đã dừng server.')